## Witaj w Labie 3, Tydzień 1 Dzień 4

Dziś zbudujemy coś z natychmiastową wartością! To początek labu, który potrwa 2 dni.

I zbudujemy ręcznie Agent Loop bez żadnego Agent Frameworka..

### Najpierw trochę przygotowań

W folderze `twin` umieściłem jeden plik `linkedin.pdf` - to pobrany PDF mojego profilu LinkedIn.

Zastąp go swoim! Powinieneś móc pobrać go ze swojego profilu LinkedIn; wejdź na stronę swojego profilu i użyj menu pod swoim nazwiskiem. Jeśli nie masz dostępu do tej funkcji, świetnie sprawdzi się dowolny PDF, np. Twoje CV.

Zrobiłem też plik o nazwie `summary.txt` w `twin` - przeczytaj go i zaktualizuj tak, żeby odzwierciedlał Ciebie.

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">Wyszukiwanie pakietów</h2>
            <span style="color:#00bfff;">W tym labie użyjemy wspaniałego pakietu Gradio do budowania szybkich UI, 
            a także popularnego czytnika PDF PyPDF. Jeśli zastanawiasz się, jak wybierać pakiety do swoich własnych projektów, zobacz Q37 na stronie <a href="https://edwarddonner.com/avatar?q=37">FAQ</a>.
            </span>
        </td>
    </tr>
</table>

In [ ]:
# Jeśli nie wiesz, co robi któryś z tych pakietów - zawsze możesz poprosić ChatGPT o wyjaśnienie!

from dotenv import load_dotenv
from openai import OpenAI
from pypdf import PdfReader
from IPython.display import Markdown, display
import gradio as gr
import json

In [ ]:
load_dotenv(override=True)
openai = OpenAI()

In [ ]:
reader = PdfReader("twin/linkedin.pdf")
linkedin = ""
for page in reader.pages:
    text = page.extract_text()
    if text:
        linkedin += text

In [ ]:
print(linkedin)

In [ ]:
with open("twin/summary.txt", "r", encoding="utf-8") as f:
    summary = f.read()

In [ ]:
print(summary)

## Dygresja: Trzy pojęcia jako przypomnienie

1. System Prompt: część danych wejściowych do LLM, która opisuje ogólny kontekst rozmowy

2. Historia konwersacji: cała dotychczasowa rozmowa

3. Iluzja pamięci: każda wiadomość do LLM jest bezstanowa. Przekazujemy całą dotychczasową rozmowę, żeby stworzyć iluzję, że model pamięta, co powiedziano 30 sekund temu...

__Więcej na ten temat w moim towarzyszącym kursie AI Engineer Core Track (pierwszy tydzień)__

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

In [ ]:
messages = [
    {"role": "system", "content": "You are a snarky, witty assistant"},
    {"role": "user", "content": "Hi, my name is Ed"},
    {"role": "assistant", "content": "Well hi there, Ed. It's nice to meet you."},
    {"role": "user", "content": "What's my name?"}
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-nano", messages=messages)
print(response.choices[0].message.content)

## Wracamy do głównego wątku!

Mamy profil LinkedIn w zmiennej `linkedin`

Mamy podsumowanie w zmiennej `summary`

Skonstruujmy System Prompt..

In [ ]:
system_prompt = f"""

# Your role

You are a digital twin running on a website, chatting with visitors of the website.
You represent the person who's website you are on.
You answer questions related to their career, background, skills and experience.

Here are the details of the person you are representing:

{summary}

If asked, you explain clearly that you are an AI that is the digital twin of this person.

# Context

Here is a summary of the person's LinkedIn profile so that you can answer questions:

{linkedin}

# Rules

Engage with the user. Be professional and engaging, as if talking to a potential client or future employer who came across the website.
Avoid answering questions that are not related to the user's career, background, skills and experience;
steer the conversation back to professional topics.

Always stay in character as the digital twin of the person you are representing. Represent the person.

IMPORTANT: If you don't know the answer, say so. Never make up an answer.
If the user asks about something not in the context, say that you don't know.
"""

In [ ]:
display(Markdown(system_prompt))

In [ ]:
messages = [
    {"role": "system", "content": system_prompt},
    {"role": "user", "content": "Hi - please tell me about yourself"},
]

In [ ]:
response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
display(Markdown(response.choices[0].message.content))

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages)
    return response.choices[0].message.content

In [ ]:
chat("Please summarize who you are", [])

## UWAGA dla tych, którzy nie używają modeli OpenAI

Jeśli używasz modeli innych niż OpenAI, może być konieczne wstawienie tej linii na początku chat():

```python
history = [{"role": h["role"], "content": h["content"]} for h in history]
```

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# A teraz - NARZĘDZIA!

Zacznijmy od funkcji...

In [ ]:
def record_email_tool(email):
    print(f"Tool called to record an email: {email}")
    with open("emails.txt", "a", encoding="utf-8") as f:
        f.write(email + "\n")
    return "Email received"

In [ ]:
record_email_tool("test@testy.com")

## Krok 1 - napisz json opisujący narzędzie


In [ ]:
record_email_tool_json = {
    "name": "record_email_tool",
    "description": "Use this tool to record that a user provided their email address",
    "parameters": {
        "type": "object",
        "properties": {
            "email": {"type": "string", "description": "The email address of this user"}
        },
        "required": ["email"],
        "additionalProperties": False
    }
}


In [ ]:
tools = [{"type": "function", "function": record_email_tool_json}]

In [ ]:
tools

## Krok 2 - nowa funkcja chat()

Tutaj implementujemy wywołanie narzędzia.

W rzeczywistości jest to trochę toporne. To jak zobaczenie składników wykwintnego przepisu i odkrycie, że te składniki są całkiem zwyczajne.

Wywoływanie narzędzi to instrukcja "if". W tym przypadku zakodowaliśmy na sztywno wszystko, zakładając, że jedynym narzędziem jest narzędzie do maili.

DYGRESJA: Jeśli myślisz - ale czekaj! Powinienem to zapamiętać, żeby móc zrobić to sam! To kluczowy punkt jest taki: to właśnie tym zajmują się za Ciebie Agent Frameworki. W praktyce prawdopodobnie nigdy więcej sam tego nie napiszesz. Jesteśmy osłonięci przed tymi instrukcjami if przez Agent Framework. Dlatego często są one opisywane jako "warstwy abstrakcji".

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    if response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            tool_call = message.tool_calls[0]
            email = json.loads(tool_call.function.arguments).get("email")
            record_email_tool(email)
            messages.append(message)
            messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

## Krok 3

Nasz pierwszy w historii Agent Loop, zrobiony bez Agent Frameworka!

Zmiany:
1. Zamiast zawsze zakładać, że jest tylko 1 wywołanie narzędzia, iterujemy po narzędziach pętlą for
2. Zmieniono `if finish_reason=="tool_calls"` na `while finish_reason=="tool_calls"`

In [ ]:
def chat(message, history):
    messages = [{"role": "system", "content": system_prompt}] + history + [{"role": "user", "content": message}]
    response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
         
    while response.choices[0].finish_reason=="tool_calls":
            message = response.choices[0].message
            messages.append(message)
            for tool_call in message.tool_calls:
                email = json.loads(tool_call.function.arguments).get("email")
                record_email_tool(email)
                messages.append({"role": "tool", "content": "Email recorded", "tool_call_id": tool_call.id})
            response = openai.chat.completions.create(model="gpt-5.4-mini", messages=messages, tools=tools)
            
    return response.choices[0].message.content

In [ ]:
gr.ChatInterface(chat).launch(inbrowser=True)

# Gratulacje!

Właśnie zaimplementowałeś Asystenta AI z Narzędziami.  
I ręcznie skręciłeś Agent Loop, bez potrzeby Agent Frameworka.  
To wszystko!

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Ćwiczenie</h2>
            <span style="color:#ff7800;">1. Dodaj wiele wywołań LLM! Po tym, jak LLM sformułuje odpowiedź, użyj kolejnego wywołania LLM, żeby ocenić, czy odpowiedź dotyczy ściśle tylko spraw zawodowych.<br/><br/>2. Zastosuj to w swoim biznesie! Zrób Asystenta AI, który potrafi odpowiadać na pytania o Twój obszar biznesowy i użyj narzędzia do zapisywania adresów mailowych osób, które chcą się skontaktować.
            </span>
        </td>
    </tr>
</table>